## Adding to previous functionality, And a tool is called manually

In [63]:
from pydantic_settings import BaseSettings, SettingsConfigDict
from pydantic import Field, SecretStr
from pydantic import BaseModel, ValidationError
import json 
from functools import wraps

class AppSettings(BaseSettings):
    model_config = SettingsConfigDict(env_file="../.env")
    groq_api_key: SecretStr

In [64]:
settings = AppSettings()   # reads from .env / environment automatically
print(settings.groq_api_key)      

**********


In [65]:
def call_groq(question: str) -> str:
    from openai import OpenAI

    client = OpenAI(api_key=settings.groq_api_key.get_secret_value(), base_url="https://api.groq.com/openai/v1")
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile", max_tokens=200, messages=[{"role": "user", "content": question}]
    )
    return response.choices[0].message.content

### The expected structure from LLM
* City name along with weather information.
* Fahrenheit instead of Celsius.

In [66]:
class WeatherInfo(BaseModel):
    city: str
    wants_fahrenheit: bool = False

# Langchain Stype Tool Creation Decorator

In [100]:
import inspect
from typing import Callable, Any, Dict
from pydantic import create_model, BaseModel

class FuncationMetadataTool:
    """Wraps a Python function with metadata for an LLM."""
    def __init__(self, func: Callable, name: str, description: str, args_schema: type[BaseModel]):
        self.func = func
        self.name = name
        self.description = description
        self.args_schema = args_schema

    def __call__(self, *args, **kwargs) -> Any:
        # Validates arguments against the Pydantic schema before execution
        validated_args = self.args_schema(**kwargs)
        return self.func(**validated_args.model_dump())

    def get_llm_schema(self) -> Dict[str, Any]:
        """Generates OpenAI-style tool definition schema."""
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": self.args_schema.model_json_schema()
            }
        }

def tool(func: Callable) -> CustomTool:
    """Decorator to transform a function into a CustomTool."""
    # 1. Extract name and description
    name = func.__name__
    description = func.__doc__ or "No description provided."
    
    # 2. Extract function signatures and type hints
    sig = inspect.signature(func)
    fields = {}
    
    for param_name, param in sig.parameters.items():
        if param_name == 'self':
            continue
        # Default to Any if no type hint is provided
        param_type = param.annotation if param.annotation != inspect.Parameter.empty else Any
        # Handle default values
        default_value = param.default if param.default != inspect.Parameter.empty else ...
        fields[param_name] = (param_type, default_value)
    
    # 3. Dynamically create a Pydantic model for input validation
    schema_name = f"{name} InputSchema"
    args_schema = create_model(schema_name, **fields)
    
    return FuncationMetadataTool(func, name, description, args_schema)


### Dummy Weather Tool, in reality information will be extracted from API's

In [101]:
@tool
def get_weather_information(city: str):
    """
    Retrieve weather information for a supported city.
    Args:
        city (str): The name of the city.
    Returns:
        dict: A dictionary containing:
            - celsius (int): Temperature in degrees Celsius.
            - conditions (str): A brief description of the weather.
    """
    weather = {
        "tokyo": {"celsius": 22, "conditions": "partly cloudy"},
        "delhi": {"celsius": 34, "conditions": "clear skies"},
        "london": {"celsius": 15, "conditions": "light rain"},
    }
    return weather.get(city.lower())

In [102]:
print(json.dumps(get_weather_information.get_llm_schema(), indent=2))

{
  "type": "function",
  "function": {
    "name": "get_weather_information",
    "description": "\nRetrieve weather information for a supported city.\nArgs:\n    city (str): The name of the city.\nReturns:\n    dict: A dictionary containing:\n        - celsius (int): Temperature in degrees Celsius.\n        - conditions (str): A brief description of the weather.\n",
    "parameters": {
      "properties": {
        "city": {
          "title": "City",
          "type": "string"
        }
      },
      "required": [
        "city"
      ],
      "title": "get_weather_information InputSchema",
      "type": "object"
    }
  }
}


### LLM with instruction and create Pydantic schema
* Asks the model to respond with only a JSON object matching WeatherInfo Model. 
* Returns the validated model on success, or a short error string if the reply didn't match.

In [76]:
def extract_weather_question(user_message: str) -> WeatherInfo | str:
    instruction = (
        "Read the user's message and reply with ONLY a JSON object -- no other "
        'text -- in this exact shape: {"city": "<city name>", '
        '"wants_fahrenheit": <true or false>}. '
        f"User's message: {user_message!r}"
    )
    raw_reply = call_groq(instruction)

    try:
        cleaned = raw_reply.strip().removeprefix("```json").removesuffix("```").strip()
        data = json.loads(cleaned)
        return WeatherInfo(** data)
    except (json.JSONDecodeError, ValidationError) as exc:
        return f"Rejected: {exc}"

#### Manual Pipeline
* Extract a trusted city with Pydantic
* Call get_weather only the extract the city.
* Pass the extracted city, here nothing is decided by the model beyond the city extraction step.

In [ ]:
def answer_weather_question(user_message: str) -> str:
    extracted = extract_weather_question(user_message)
    print(f"Extracted: {extracted!r}")
    # extracted is of Type WeatherInfo
    if not isinstance(extracted, WeatherInfo):
        return f"Could not extract a city: {extracted}"
    return get_weather_information(city = extracted.city.lower())

In [ ]:
print(answer_weather_question("What's the weather like in Tokyo right now?"))

Extracted: WeatherInfo(city='Tokyo', wants_fahrenheit=False)
{'celsius': 22, 'conditions': 'partly cloudy'}
